In [ ]:
# install libraries
!pip install requests pandas nltk textblob vaderSentiment beautifulsoup4 matplotlib seaborn wordcloud plotly

## NEWS API
[text](https://newsapi.org/register)

Account detail
bala.sbm.kk@gmail.com
Demo@1234

api key
f18eab0734014db589aa291c1f6ee071

In [1]:
# Download NLTK data

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng')

print("✅ All dependencies installed and ready!")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\subramm6\AppData\Roaming\nltk_data...


✅ All dependencies installed and ready!


[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


In [4]:
# ── Core libraries ──────────────────────────────────────────────
import requests
import json
import re
import time
from datetime import datetime

# ── Data handling ────────────────────────────────────────────────
import pandas as pd

# ── NLP ──────────────────────────────────────────────────────────

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


# ── Web Scraping ─────────────────────────────────────────────────
from bs4 import BeautifulSoup

# ── Visualization ─────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('imported successfully...')

imported successfully...


In [1]:
NEWS_API_KEY = "f18eab0734014db589aa291c1f6ee071"
CATEGORIES = ["technology", "business"]
SEARCH_QUERIES = ["economy", "artificial intelligence"]
PAGE_SIZE = 1
COUNTRY ="us"
BASE_URL = "https://newsapi.org/v2"

In [5]:
all_articles = []
def fetch_top_headlines(category:str, country:str, page_size:int=PAGE_SIZE) -> list:
   url = f"{BASE_URL}/top-headlines"
   print(url)
   request = {
       "apiKey" : NEWS_API_KEY,
       "category": category,
       "country": country,
       "pageSize" : page_size
   }
   print(request)
   response = requests.get(url, params=request, timeout=10)
   print(response)
   if(response.status_code != 200):
       print(f" API error {response.status_code}")
   articles = response.json().get('articles', [])
   print(articles)
   return articles


for cat in CATEGORIES:
   articles = fetch_top_headlines(category=cat, country=COUNTRY)
   for art in articles:
       art["query_category"] = cat
   all_articles.extend(articles)
print(all_articles)


https://newsapi.org/v2/top-headlines
{'apiKey': 'f18eab0734014db589aa291c1f6ee071', 'category': 'technology', 'country': 'us', 'pageSize': 1}
<Response [200]>
[{'source': {'id': None, 'name': 'My Nintendo News'}, 'author': 'Sickr', 'title': 'Persona 4 Revival due to be completed end of August and release early 2027 - My Nintendo News', 'description': 'Tose is the Japanese third party studio which is co-developing the long-awaited Persona 4 Revival from Atlus. In a recent finical report, which shows the company’s progress on five projects…', 'url': 'https://mynintendonews.com/2026/05/04/persona-4-revival-due-to-be-completed-end-of-august-and-release-early-2027/', 'urlToImage': 'https://i0.wp.com/mynintendonews.com/wp-content/uploads/2022/12/persona_4_golden_art.jpeg?fit=1280%2C720&ssl=1', 'publishedAt': '2026-05-04T13:36:51Z', 'content': 'Tose is the Japanese third party studio which is co-developing the long-awaited Persona 4 Revival from Atlus. In a recent finical report, which shows 

In [6]:
def fetch_by_keyword(query:str, page_size:int=PAGE_SIZE) -> list:
   url = f"{BASE_URL}/everything"
   print(url)
   request = {
       "apiKey" : NEWS_API_KEY,
       "q": query,
       "language": "en",
       "sortBy": "publishedAt",
       "pageSize": page_size
   }
   print(request)
   response = requests.get(url, params=request, timeout=10)
   print(response)
   if(response.status_code != 200):
       print(f" API error {response.status_code}")
   articles = response.json().get('articles', [])
   print(articles)
   return articles


for query in SEARCH_QUERIES:
   articles = fetch_by_keyword(query=query)
   for art in articles:
       art["query_category"] = f"kw:{query}"
   all_articles.extend(articles)


for art in all_articles:
   print(art)

https://newsapi.org/v2/everything
{'apiKey': 'f18eab0734014db589aa291c1f6ee071', 'q': 'economy', 'language': 'en', 'sortBy': 'publishedAt', 'pageSize': 1}
<Response [200]>
[{'source': {'id': None, 'name': 'Theflightdeal.com'}, 'author': 'The Flight Deal', 'title': 'Asiana Airlines: Seattle – Ho Chi Minh City, Vietnam. $740. Roundtrip, including all Taxes', 'description': 'A good sale to VietnamVietnam introduced a new e-Visa system - take advantage of it. Its the cheapest option.Matrix Airfare Search will price this at $752. Use those dates on Priceline should', 'url': 'https://www.theflightdeal.com/2026/05/04/asiana-airlines-seattle-ho-chi-minh-city-vietnam-740-roundtrip-including-all-taxes/', 'urlToImage': 'https://www.theflightdeal.com/wp-content/uploads/2014/08/ho_chi_minh_city_opera_house-640x390.jpg', 'publishedAt': '2026-05-04T17:03:38Z', 'content': 'A good sale to Vietnam\r\nVietnam introduced a new e-Visa system take advantage of it. Its the cheapest option.\r\nMatrix Airfare 

In [7]:
def articles_to_dataframe(articles:list) -> pd.DataFrame:
   rows = []
   for art in articles:
       rows.append({
           "title": art.get("title") or "",
           "description" : art.get("description") or "",
           "source" : art.get("source", {}).get("name") or "Unknown",
           "url": art.get("url") or "",
           "published_at" : art.get("publishedAt") or "",
           "category" : art.get("query_category"),
           "full_text" : " ".join(filter(None, [
               art.get("title") or "",
               art.get("description") or "",
               art.get("content") or ""
           ]))
       })
   df = pd.DataFrame(rows)
   return df


df = articles_to_dataframe(all_articles)
df.head()

,title,description,source,url,published_at,category,full_text
0,Persona 4 Revival due to be completed end of A...,Tose is the Japanese third party studio which ...,My Nintendo News,https://mynintendonews.com/2026/05/04/persona-...,2026-05-04T13:36:51Z,technology,Persona 4 Revival due to be completed end of A...
1,"Asiana Airlines: Seattle – Ho Chi Minh City, V...",A good sale to VietnamVietnam introduced a new...,Theflightdeal.com,https://www.theflightdeal.com/2026/05/04/asian...,2026-05-04T17:03:38Z,kw:economy,"Asiana Airlines: Seattle – Ho Chi Minh City, V..."
2,ENESS creates a conveyor-belt AI fantasy where...,visitors place their phones onto a moving conv...,Designboom,https://www.designboom.com/art/eness-conveyor-...,2026-05-04T17:00:08Z,kw:artificial intelligence,ENESS creates a conveyor-belt AI fantasy where...
